In [ ]:
# ===============================================
# In This file:
# The Single-generator R-GAN framework is implemented on the CIFAR-10 dataset for Targeted Attack (target = class 0), according the R-GAN paper.
# The robustness (accuracy) for classifiers of R-net, C-net, and T-net are evaluated against attacks generated by G-net, FGSM, PGD, and C&W.
# ===============================================

In [ ]:
# === Change this PATH to actual the path of CIFAR-10 dataset ===
dataset_path = r"C:\Users\Administrator\Desktop\R-GAN\Dataset\cifar-10-python"

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [ ]:

# Define transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])



# Load CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=True,
    transform=transform_train,
    download=False
)

test_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=False,
    transform=transform_test,
    download=False
)

batch_size = 128
num_workers = 2 if torch.cuda.is_available() else 0

train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(" CIFAR-10 loaded successfully!")
print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")


In [ ]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# CIFAR-10 normalization
CIFAR10_MEAN = mean = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = std = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

def denormalize(imgs_norm):
    """imgs_norm: normalized (mean/std) -> returns imgs in [0,1] pixel space"""
    return imgs_norm * CIFAR10_STD + CIFAR10_MEAN

def renormalize(imgs_01):
    """imgs_01 in [0,1] -> normalized"""
    return (imgs_01 - CIFAR10_MEAN) / CIFAR10_STD

# L_inf epsilon in pixel space
eps = 8.0 / 255.0
print("eps (L_inf):", eps)


Device: cuda
eps (L_inf): 0.03137254901960784


In [ ]:
# ===============================================
# In This Cell:
# The architecture of the classifiers f-net (C-net) and R-net is defined as Wide ResNet-34.
# The pretrained classifier C-net described in the paper is referred to as f-net in the code (f-net in code = C-net in the paper).
# ===============================================

class BasicBlockWide(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1, dropRate=0.0):
        super(BasicBlockWide, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = (in_planes == out_planes)
        self.shortcut = (not self.equalInOut) and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False) or None

    def forward(self, x):
        if not self.equalInOut:
            x = self.relu1(self.bn1(x))
        else:
            out = self.relu1(self.bn1(x))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
        if self.droprate > 0:
            out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.shortcut is None else self.shortcut(x), out)


class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super(NetworkBlock, self).__init__()
        layers = []
        for i in range(nb_layers):
            layers.append(block(i == 0 and in_planes or out_planes, out_planes,
                                i == 0 and stride or 1, dropRate))
        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


class WideResNet_CIFAR(nn.Module):
    def __init__(self, depth=34, widen_factor=10, num_classes=10, dropRate=0.0):
        super(WideResNet_CIFAR, self).__init__()
        nChannels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        assert ((depth - 4) % 6 == 0), 'Depth should be 6n+4'
        n = (depth - 4) // 6
        block = BasicBlockWide

        self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], block, 1, dropRate)
        self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], block, 2, dropRate)
        self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], block, 2, dropRate)
        self.bn1 = nn.BatchNorm2d(nChannels[3])
        self.relu = nn.ReLU(inplace=True)
        self.fc = nn.Linear(nChannels[3], num_classes)

        # Initialize weights
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.avg_pool2d(out, 8)
        out = out.view(out.size(0), -1)
        return self.fc(out)


def WideResNet34_10_CIFAR():
    """Return WRN-34-10 (depth=34, widen_factor=10)"""
    return WideResNet_CIFAR(depth=34, widen_factor=10, num_classes=10)


In [ ]:
# ===============================================
# In This Cell:
# The architecture of the generator is defined.
# The generator is denoted as G in the code and as G-net in the paper (G in the code = G-net in the paper).
# ===============================================


class ResnetBlock(nn.Module):
    def __init__(self, dim, norm_layer=nn.BatchNorm2d, use_dropout=False, use_bias=False):
        super().__init__()
        conv_block = [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim),
            nn.ReLU(True)
        ]
        if use_dropout:
            conv_block += [nn.Dropout(0.5)]
        conv_block += [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim)
        ]
        self.conv_block = nn.Sequential(*conv_block)

    def forward(self, x):
        return x + self.conv_block(x)

class Generator(nn.Module):
    def __init__(self, gen_input_nc=3, image_nc=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(gen_input_nc, 32, 3, 1, 1), nn.InstanceNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, 2, 1),           nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, 2, 1),          nn.InstanceNorm2d(128), nn.ReLU(True),
        )
        self.bottleneck = nn.Sequential(ResnetBlock(128), ResnetBlock(128), ResnetBlock(128))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),  nn.InstanceNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, image_nc, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, x_01):
        # x_01 in [0,1]
        return self.decoder(self.bottleneck(self.encoder(x_01)))


In [ ]:
# ===============================================
# In This Cell:
# The classifier f-net (C-net) is trained using Wide ResNet-34.
# ===============================================


f_net = WideResNet34_10_CIFAR().to(device)
criterion = nn.CrossEntropyLoss()
optimizer_f = optim.SGD(f_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_f = optim.lr_scheduler.MultiStepLR(optimizer_f, milestones=[80, 120], gamma=0.1)

epochs_f = 150
print("Training f_net for", epochs_f, "epochs...")

for epoch in range(1, epochs_f+1):
    # Train
    f_net.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs_norm, labels in train_dataloader:
        imgs_norm, labels = imgs_norm.to(device), labels.to(device)
        optimizer_f.zero_grad()
        logits = f_net(imgs_norm)  # imgs_norm already normalized by transform_train
        loss = criterion(logits, labels)
        loss.backward()
        optimizer_f.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * correct / total
    train_loss = running_loss / len(train_dataloader)

    # Test
    if (epoch ==1 or epoch % 10 == 0 or epoch==epochs_f):

        f_net.eval()
        correct_test = 0
        total_test = 0
        test_loss = 0.0
        with torch.no_grad():
            for imgs_t_norm, labels_t in test_dataloader:
                imgs_t_norm, labels_t = imgs_t_norm.to(device), labels_t.to(device)
                logits_t = f_net(imgs_t_norm)
                loss_t = criterion(logits_t, labels_t)
                test_loss += loss_t.item()
                preds_t = logits_t.argmax(dim=1)
                correct_test += (preds_t == labels_t).sum().item()
                total_test += labels_t.size(0)
        test_acc = 100.0 * correct_test / total_test
        test_loss = test_loss / len(test_dataloader)
        print(f"[f_train] Epoch {epoch}/{epochs_f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")
    scheduler_f.step()

# Freeze f_net and set eval
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False


In [ ]:
# ===============================================
# In This Cell:
# The R-GAN Framework is trained.
# R-net and G(G-net) are trained together competitively in the R-GAN Framework.
# The R-GAN Framework in this cell and file is implemented for the Targeted Attack scenario.
# Generator G forces both f-net & R-net classifiers towards the class 0 category (target = class 0).
# ===============================================

# Build R_net, G
R_net = WideResNet34_10_CIFAR().to(device)
optimizer_R = optim.SGD(R_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

# --- Scheduler for R_net (same schedule as f_net used earlier) ---
scheduler_R = optim.lr_scheduler.MultiStepLR(optimizer_R, milestones=[80, 120], gamma=0.1)


G = Generator(gen_input_nc=3, image_nc=3).to(device)
optimizer_G = torch.optim.Adam(G.parameters(), lr=1e-3)

# R-GAN hyperparams
lambda_f = 5.0              # Coefficient for fooling pretrained f-net during generator G update
lambda_R = 5.0              # Coefficient for fooling R-net during generator G update
lambda_pert = 1.0           # Coefficient for perturbation L2 regularizer during generator G update
margin = 0.0                # hinge margin for adversarial hinge on logits
num_epochs = 150
print_every = 1
print("Start")

target_class = 0  # targeted class (force predictions to be this label)

def hinge_adv_loss_logits(logits, labels, margin=0.0):
    B = logits.shape[0]
    true_logits = logits[torch.arange(B), labels]
    other_logits = logits.clone()
    other_logits[torch.arange(B), labels] = -1e9
    max_other, _ = other_logits.max(dim=1)
    loss = torch.clamp(true_logits - max_other + margin, min=0.0)
    return loss.mean()

for epoch in range(1, num_epochs+1):
    G.train(); R_net.train()
    total_loss_R = total_loss_G = total_loss_f = total_loss_pert = 0.0
    batches = 0

    for imgs_norm, labels in train_dataloader:
        imgs_norm, labels = imgs_norm.to(device), labels.to(device)
        batches += 1

        # convert normalized images -> pixel space [0,1] for generator input
        imgs_pixel = denormalize(imgs_norm)

        # -------------------------
        # 1) R-update: train R on clean + adv (G frozen)
        # -------------------------
        G.eval()
        with torch.no_grad():
            delta = G(imgs_pixel)                 # raw G output in [-1,1] from Tanh
            delta = torch.clamp(delta, -eps, eps) # clamp to pixel-space bound
            adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
            adv_norm = renormalize(adv_pixel)     # normalized for classifier input

        R_net.train()
        logits_clean = R_net(imgs_norm)   # normalized clean
        logits_adv   = R_net(adv_norm)
        # R_net learns to classify both clean and adv correctly
        loss_R = F.cross_entropy(logits_clean, labels) + F.cross_entropy(logits_adv, labels)

        optimizer_R.zero_grad()
        loss_R.backward()
        optimizer_R.step()
        total_loss_R += loss_R.item()

        # -------------------------
        # 2) G-update: train G to enforce target label 0 on both f_net and R_net (freeze R params)
        # -------------------------
        G.train()
        R_net.eval()
        for p in R_net.parameters():
            p.requires_grad = False

        # target_labels tensor filled with the chosen target class (0)
        target_labels = torch.full_like(labels, fill_value=target_class, device=device)

        delta = G(imgs_pixel)
        delta = torch.clamp(delta, -eps, eps)
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)

        # targeted loss: encourage f_net(adv) == target_class and R_net(adv) == target_class
        logits_f = f_net(adv_norm)
        loss_f_target = F.cross_entropy(logits_f, target_labels)

        logits_R_adv = R_net(adv_norm)
        loss_R_target = F.cross_entropy(logits_R_adv, target_labels)

        # perturbation regularizer (mean L2)
        loss_pert = torch.mean(torch.norm(delta.view(delta.size(0), -1), dim=1, p=2))

        # Combine losses: lambda_f and lambda_R weigh fooling f and R respectively.
        loss_G = lambda_f * loss_f_target + lambda_R * loss_R_target + lambda_pert * loss_pert

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

        total_loss_G += loss_G.item()
        total_loss_f += loss_f_target.item()
        total_loss_pert += loss_pert.item()

        for p in R_net.parameters():
            p.requires_grad = True

    # epoch summary
    avg_R = total_loss_R / batches
    avg_G = total_loss_G / batches
    avg_f = total_loss_f / batches
    avg_pert = total_loss_pert / batches

    if epoch % print_every == 0:
        print(f"[Epoch {epoch}/{num_epochs}] Loss_R: {avg_R:.4f} | Loss_G: {avg_G:.4f} | Loss_f_target: {avg_f:.4f} | Loss_pert(L2): {avg_pert:.4f}")

    # Evalaluation: clean & adv-by-current-G
    if (epoch == 1 or epoch % 2 == 1 or epoch == num_epochs):
        G.eval(); R_net.eval(); f_net.eval()
        correct_clean_R = correct_adv_R = 0
        correct_clean_f = correct_adv_f = 0
        total_test = 0
        with torch.no_grad():
            for imgs_t_norm, labels_t in test_dataloader:
                imgs_t_norm, labels_t = imgs_t_norm.to(device), labels_t.to(device)
                total_test += labels_t.size(0)

                preds_R_clean = R_net(imgs_t_norm).argmax(dim=1)
                preds_f_clean = f_net(imgs_t_norm).argmax(dim=1)
                correct_clean_R += (preds_R_clean == labels_t).sum().item()
                correct_clean_f += (preds_f_clean == labels_t).sum().item()

                imgs_t_pixel = denormalize(imgs_t_norm)
                delta_t = torch.clamp(G(imgs_t_pixel), -eps, eps)
                adv_t_pixel = torch.clamp(imgs_t_pixel + delta_t, 0.0, 1.0)
                adv_t_norm = renormalize(adv_t_pixel)

                preds_R_adv = R_net(adv_t_norm).argmax(dim=1)
                preds_f_adv = f_net(adv_t_norm).argmax(dim=1)
                correct_adv_R += (preds_R_adv == labels_t).sum().item()
                correct_adv_f += (preds_f_adv == labels_t).sum().item()

        print(f" Eval - R clean acc: {100*correct_clean_R/total_test:.2f}% | R adv acc: {100*correct_adv_R/total_test:.2f}%")
        print(f"      f clean acc: {100*correct_clean_f/total_test:.2f}% | f adv acc: {100*correct_adv_f/total_test:.2f}%")
   # --- step scheduler for R_net every epoch ---
    scheduler_R.step()

Start
[Epoch 1/150] Loss_R: 3.5621 | Loss_G: 162.9025 | Loss_f_target: 9.2286 | Loss_pert(L2): 1.7373
 Eval - R clean acc: 41.72% | R adv acc: 40.63%
      f clean acc: 95.88% | f adv acc: 95.25%
[Epoch 2/150] Loss_R: 2.5500 | Loss_G: 68.9647 | Loss_f_target: 9.2045 | Loss_pert(L2): 1.7385
[Epoch 3/150] Loss_R: 1.8471 | Loss_G: 72.9787 | Loss_f_target: 9.1466 | Loss_pert(L2): 1.7384
 Eval - R clean acc: 66.33% | R adv acc: 64.46%
      f clean acc: 95.88% | f adv acc: 94.06%
[Epoch 4/150] Loss_R: 1.4527 | Loss_G: 75.7718 | Loss_f_target: 9.1286 | Loss_pert(L2): 1.7383
[Epoch 5/150] Loss_R: 1.2263 | Loss_G: 78.3322 | Loss_f_target: 9.1061 | Loss_pert(L2): 1.7384
 Eval - R clean acc: 76.22% | R adv acc: 74.48%
      f clean acc: 95.88% | f adv acc: 93.92%
[Epoch 6/150] Loss_R: 1.1139 | Loss_G: 79.4709 | Loss_f_target: 9.1090 | Loss_pert(L2): 1.7384
[Epoch 7/150] Loss_R: 1.0352 | Loss_G: 80.9084 | Loss_f_target: 9.1223 | Loss_pert(L2): 1.7385
 Eval - R clean acc: 81.47% | R adv acc: 81.20

In [ ]:
# ===============================================
# In the three cells below:
# T-net is trined using Model ResNet-18.
# To train T-net with Model ResNet-32 or Model Wide ResNet-34, as mentioned in the R-GAN Paper, set T-net according to the architecture of these models.
# The Architecture of Model ResNet-32 or Wide ResNet-34 is defined in the "Architecture of Models" file.
# ===============================================

In [ ]:
# Shuffle the train dataset
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
print("The Train Dataset is shuffled in order difference with Victim Model")

In [ ]:
# ===============================================
# In This Cell:
# The Architecture of ResNet-18 (T-net) is defined (CIFAR-10 version).
# ===============================================


# Basic residual block
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

# ResNet-18 adapted for CIFAR-10
class ResNet_CIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet_CIFAR, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)  # global avg pool
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18_CIFAR():
    return ResNet_CIFAR(BasicBlock, [2,2,2,2])


In [ ]:

# ===============================================
# In This Cell:
# The classifier of T-net is trained.
# ===============================================

import torch
import torch.nn as nn
import torch.optim as optim

# build T_net
T_net = ResNet18_CIFAR().to(device)

# loss / optimizer / scheduler (same as f_net)
criterion_t = nn.CrossEntropyLoss()
optimizer_t = optim.SGD(T_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_t = optim.lr_scheduler.MultiStepLR(optimizer_t, milestones=[80, 120], gamma=0.1)

epochs_t = 150
print("Training T_net (ResNet18) for", epochs_t, "epochs...")

for epoch in range(1, epochs_t + 1):
    # ---- Train ----
    T_net.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs_norm, labels in train_dataloader:
        imgs_norm = imgs_norm.to(device)
        labels = labels.to(device)

        optimizer_t.zero_grad()
        logits = T_net(imgs_norm)           # imgs_norm assumed normalized by the transforms
        loss = criterion_t(logits, labels)
        loss.backward()
        optimizer_t.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * correct / total if total > 0 else 0.0
    train_loss = running_loss / (len(train_dataloader) if len(train_dataloader)>0 else 1)

    # ---- Test / eval periodically ----
    if (epoch == 1 or epoch % 10 == 0 or epoch == epochs_t):
        T_net.eval()
        correct_test = 0
        total_test = 0
        test_loss = 0.0
        with torch.no_grad():
            for imgs_t_norm, labels_t in test_dataloader:
                imgs_t_norm = imgs_t_norm.to(device)
                labels_t = labels_t.to(device)
                logits_t = T_net(imgs_t_norm)
                loss_t = criterion_t(logits_t, labels_t)
                test_loss += loss_t.item()
                preds_t = logits_t.argmax(dim=1)
                correct_test += (preds_t == labels_t).sum().item()
                total_test += labels_t.size(0)

        test_acc = 100.0 * correct_test / total_test if total_test>0 else 0.0
        test_loss = test_loss / (len(test_dataloader) if len(test_dataloader)>0 else 1)
        print(f"[T_train] Epoch {epoch}/{epochs_t} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

    scheduler_t.step()

# Freeze T_net and set eval (like that did for f_net)
T_net.eval()
for p in T_net.parameters():
    p.requires_grad = False

print("Finished training T_net; model set to eval and frozen.")


In [ ]:
# ================================================
# In This Cell: 
# The accuracy of T_net, R_net, and f_net for perturbed data by generator G (G-net attacker) and clean data is evaluated.
# In this file, the generator G-net produces Targeted Attacks toward class 0 category (target = class 0).
# ================================================


import torch
from tqdm import tqdm
import torch.nn.functional as F

G.eval()
T_net.eval()
R_net.eval()
f_net.eval()

try:
    eps  #defined previously
except NameError:
    eps = 8.0 / 255.0

total = 0
correct_clean_T = 0
correct_adv_T = 0

correct_clean_R = 0
correct_adv_R = 0
correct_clean_f = 0
correct_adv_f = 0

with torch.no_grad():
    for imgs_norm, labels in tqdm(test_dataloader, desc="Eval T_net on clean & G-adv"):
        imgs_norm = imgs_norm.to(device)
        labels = labels.to(device)
        B = imgs_norm.size(0)
        total += B

        preds_T_clean = T_net(imgs_norm).argmax(dim=1)
        correct_clean_T += (preds_T_clean == labels).sum().item()

        preds_R_clean = R_net(imgs_norm).argmax(dim=1)
        correct_clean_R += (preds_R_clean == labels).sum().item()

        preds_f_clean = f_net(imgs_norm).argmax(dim=1)
        correct_clean_f += (preds_f_clean == labels).sum().item()

        # Build adversarial images using the trained G
        imgs_pixel = denormalize(imgs_norm)               # normalized -> pixel [0,1]
        delta = G(imgs_pixel)                             # G output (pixel-space)
        delta = torch.clamp(delta, -eps, eps)             # clamp in pixel-space
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)                 # back to normalized input for networks

        # Evaluate T_net on adv images
        preds_T_adv = T_net(adv_norm).argmax(dim=1)
        correct_adv_T += (preds_T_adv == labels).sum().item()

        #Evaluate R_net and f_net on same adv images for comparison
        preds_R_adv = R_net(adv_norm).argmax(dim=1)
        correct_adv_R += (preds_R_adv == labels).sum().item()

        preds_f_adv = f_net(adv_norm).argmax(dim=1)
        correct_adv_f += (preds_f_adv == labels).sum().item()

# compute percentages
clean_acc_T = 100.0 * correct_clean_T / total
adv_acc_T   = 100.0 * correct_adv_T   / total

clean_acc_R = 100.0 * correct_clean_R / total
adv_acc_R   = 100.0 * correct_adv_R   / total

clean_acc_f = 100.0 * correct_clean_f / total
adv_acc_f   = 100.0 * correct_adv_f   / total

print("=== Final evaluation using trained G ===")
print(f"Total test samples: {total}")
print("")
print(f"T_net - clean accuracy: {clean_acc_T:.2f}%")
print(f"T_net - G-adv accuracy : {adv_acc_T:.2f}% (drop {clean_acc_T - adv_acc_T:.2f} pp)")
print("")
# optional comparison
print(f"R_net - clean accuracy: {clean_acc_R:.2f}%")
print(f"R_net - G-adv accuracy : {adv_acc_R:.2f}% (drop {clean_acc_R - adv_acc_R:.2f} pp)")
print("")
print(f"f_net - clean accuracy: {clean_acc_f:.2f}%")
print(f"f_net - G-adv accuracy : {adv_acc_f:.2f}% (drop {clean_acc_f - adv_acc_f:.2f} pp)")


In [ ]:
#================================================================
# In the The Following Cells:
# The accuracy of R-net and T-net against Targeted Transferable Attacks crafted on f-net (C-net) using FGSM, PGD, and C&W attacks is evaluated.
# The Targeted attacks are toward class 0 (target = class 0).
# ===============================================================


In [ ]:
# ===============================================
# In This Cell:
# Targeted FGSM attacks (target = class 0) crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against targeted FGSM attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval(); R_net.eval(); T_net.eval()

eps_pixel = 8.0 / 255.0
target_label = 0

# normalized-space epsilon / clamps
eps_norm = (eps_pixel / std).to(device)
min_norm = ((0.0 - mean) / std).to(device)
max_norm = ((1.0 - mean) / std).to(device)

total = 0
correct_clean_f = correct_clean_R = correct_clean_T = 0
correct_adv_f = correct_adv_R = correct_adv_T = 0

# targeted-success counters (forced to class "0")
f_target_success = 0
transfer_R = 0
transfer_T = 0

print(f"Running TARGETED FGSM (target={target_label}) crafted on f_net ...")

for imgs_norm, labels in tqdm(test_dataloader, desc="TFGSM eval (f->transfer)"):
    imgs_norm = imgs_norm.to(device)
    labels = labels.to(device)
    B = imgs_norm.size(0)
    total += B

    target = torch.full_like(labels, target_label, device=device)

    # clean predictions
    with torch.no_grad():
        preds_f_clean = f_net(imgs_norm).argmax(dim=1)
        preds_R_clean = R_net(imgs_norm).argmax(dim=1)
        preds_T_clean = T_net(imgs_norm).argmax(dim=1)
        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # craft targeted FGSM: minimize loss towards target => step opposite sign
    imgs_norm_adv = imgs_norm.clone().detach().requires_grad_(True)
    logits = f_net(imgs_norm_adv)
    loss = F.cross_entropy(logits, target)
    loss.backward()
    grad_sign = imgs_norm_adv.grad.data.sign()

    # targeted FGSM: subtract gradient sign
    delta_norm = - eps_norm * grad_sign
    adv_norm = torch.clamp(imgs_norm + delta_norm, min_norm, max_norm)

    # eval on adv_norm
    with torch.no_grad():
        preds_f_adv = f_net(adv_norm).argmax(dim=1)
        preds_R_adv = R_net(adv_norm).argmax(dim=1)
        preds_T_adv = T_net(adv_norm).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # targeted success: f_net predicted target_label
        mask_targeted = (preds_f_adv == target)
        n_success = mask_targeted.sum().item()
        if n_success > 0:
            f_target_success += n_success
            transfer_R += ((preds_R_adv == target) & mask_targeted).sum().item()
            transfer_T += ((preds_T_adv == target) & mask_targeted).sum().item()

# final metrics
clean_acc_f = 100.0 * correct_clean_f / total
clean_acc_R = 100.0 * correct_clean_R / total
clean_acc_T = 100.0 * correct_clean_T / total

adv_acc_f = 100.0 * correct_adv_f / total
adv_acc_R = 100.0 * correct_adv_R / total
adv_acc_T = 100.0 * correct_adv_T / total

transfer_rate_R = 100.0 * transfer_R / f_target_success if f_target_success>0 else float('nan')
transfer_rate_T = 100.0 * transfer_T / f_target_success if f_target_success>0 else float('nan')

print("\n=== Targeted FGSM (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")
print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")
print(f"\nAdv (targeted FGSM -> target={target_label}) accuracy (models evaluated on adv crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}%")
print(f"  R_net: {adv_acc_R:.2f}%")
print(f"  T_net: {adv_acc_T:.2f}%")
print(f"\nSuccess forcing label={target_label} on f_net: {f_target_success} / {total}  ({100*f_target_success/total:.2f}%)")
print(f"Of those forced-to-zero, transfer -> R_net: {transfer_rate_R:.2f}% | T_net: {transfer_rate_T:.2f}%")


In [ ]:

# ===============================================
# In This Cell:
# Targeted PGD attacks (target = class 0) crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against Targeted PGD attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval(); R_net.eval(); T_net.eval()

eps_pixel = 8.0 / 255.0
alpha_pixel = 2.0 / 255.0
pgd_steps = 30
random_start = True
target_label = 0

eps_norm = (eps_pixel / std).to(device)
alpha_norm = (alpha_pixel / std).to(device)
min_norm = ((0.0 - mean) / std).to(device)
max_norm = ((1.0 - mean) / std).to(device)

total = 0
correct_clean_f = correct_clean_R = correct_clean_T = 0
correct_adv_f = correct_adv_R = correct_adv_T = 0

f_target_success = 0
transfer_R = 0
transfer_T = 0

print(f"Running TARGETED PGD (target={target_label}) on f_net (eps={eps_pixel}, alpha={alpha_pixel}, steps={pgd_steps})...")

for imgs_norm, labels in tqdm(test_dataloader, desc="TPGD eval (f->transfer)"):
    imgs_norm = imgs_norm.to(device)
    labels = labels.to(device)
    B = imgs_norm.size(0)
    total += B

    target = torch.full_like(labels, target_label, device=device)


    with torch.no_grad():
        correct_clean_f += (f_net(imgs_norm).argmax(dim=1) == labels).sum().item()
        correct_clean_R += (R_net(imgs_norm).argmax(dim=1) == labels).sum().item()
        correct_clean_T += (T_net(imgs_norm).argmax(dim=1) == labels).sum().item()

    if random_start:
        rand_delta_pixel = (torch.rand_like(imgs_norm) * 2.0 - 1.0) * eps_pixel
        rand_delta_norm = rand_delta_pixel / std
        adv = torch.clamp(imgs_norm + rand_delta_norm, min_norm, max_norm).detach()
    else:
        adv = imgs_norm.clone().detach()

    # iterative targeted steps: moving opposite gradient sign to minimize CE(target)
    for _ in range(pgd_steps):
        adv.requires_grad_(True)
        logits = f_net(adv)
        loss = F.cross_entropy(logits, target)
        loss.backward()
        grad_sign = adv.grad.data.sign()
        with torch.no_grad():
            adv = adv - alpha_norm * grad_sign                # targeted: step opposite
            adv = torch.max(torch.min(adv, imgs_norm + eps_norm), imgs_norm - eps_norm)
            adv = torch.clamp(adv, min_norm, max_norm)
        adv = adv.detach()

    adv_norm = adv

    # evaluate
    with torch.no_grad():
        preds_f_adv = f_net(adv_norm).argmax(dim=1)
        preds_R_adv = R_net(adv_norm).argmax(dim=1)
        preds_T_adv = T_net(adv_norm).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        mask_targeted = (preds_f_adv == target)
        n_success = mask_targeted.sum().item()
        if n_success > 0:
            f_target_success += n_success
            transfer_R += ((preds_R_adv == target) & mask_targeted).sum().item()
            transfer_T += ((preds_T_adv == target) & mask_targeted).sum().item()

# final metrics
clean_acc_f = 100.0 * correct_clean_f / total
clean_acc_R = 100.0 * correct_clean_R / total
clean_acc_T = 100.0 * correct_clean_T / total

adv_acc_f = 100.0 * correct_adv_f / total
adv_acc_R = 100.0 * correct_adv_R / total
adv_acc_T = 100.0 * correct_adv_T / total

transfer_rate_R = 100.0 * transfer_R / f_target_success if f_target_success>0 else float('nan')
transfer_rate_T = 100.0 * transfer_T / f_target_success if f_target_success>0 else float('nan')

print("\n=== Targeted PGD (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")
print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")
print(f"\nAdv (targeted PGD -> target={target_label}) accuracy:")
print(f"  f_net: {adv_acc_f:.2f}%")
print(f"  R_net: {adv_acc_R:.2f}%")
print(f"  T_net: {adv_acc_T:.2f}%")
print(f"\nSuccess forcing label={target_label} on f_net: {f_target_success} / {total}  ({100*f_target_success/total:.2f}%)")
print(f"Of those forced-to-zero, transfer -> R_net: {transfer_rate_R:.2f}% | T_net: {transfer_rate_T:.2f}%")


In [ ]:

# ===============================================
# In This Cell:
# Targeted C&W attacks (target = class 0) crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against Targeted C&W attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval(); R_net.eval(); T_net.eval()

cw_steps = 1000       # inner optimization steps (increase for stronger attack)
lr_cw = 0.01
c = 0.1             # L2 weight 
kappa = 0.0
target_label = 0

total = 0
correct_clean_f = correct_clean_R = correct_clean_T = 0
correct_adv_f = correct_adv_R = correct_adv_T = 0

f_target_success = 0
transfer_R = 0
transfer_T = 0

print(f"Running TARGETED C&W (target={target_label}) crafted on f_net (steps={cw_steps}, c={c}) ...")

for imgs_norm, labels in tqdm(test_dataloader, desc="TCW eval (f->transfer)"):
    imgs_norm = imgs_norm.to(device)
    labels = labels.to(device)
    B = imgs_norm.size(0)
    total += B
    target = torch.full_like(labels, target_label, device=device)

    # clean preds
    with torch.no_grad():
        correct_clean_f += (f_net(imgs_norm).argmax(dim=1) == labels).sum().item()
        correct_clean_R += (R_net(imgs_norm).argmax(dim=1) == labels).sum().item()
        correct_clean_T += (T_net(imgs_norm).argmax(dim=1) == labels).sum().item()

    # operate in pixel space for delta
    imgs_pixel = denormalize(imgs_norm)  # [0,1]

    delta = torch.zeros_like(imgs_pixel, requires_grad=True, device=device)
    opt = torch.optim.Adam([delta], lr=lr_cw)

    for step in range(cw_steps):
        opt.zero_grad()
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)

        logits = f_net(adv_norm)  # [B,C]

        tgt_logits = logits[torch.arange(B), target]
        other_logits = logits.clone()
        other_logits[torch.arange(B), target] = -1e9
        max_other, _ = other_logits.max(dim=1)

        # targeted hinge: want target_logit >= max_other + kappa -> hinge
        hinge = torch.clamp(max_other - tgt_logits + kappa, min=0.0).mean()
        l2 = (delta.view(B, -1).pow(2).sum(dim=1)).mean()
        loss = hinge + c * l2

        loss.backward()
        opt.step()

        # optional early stop: if all samples already classified as target by f_net
        with torch.no_grad():
            preds_now = f_net(renormalize(torch.clamp(imgs_pixel + delta,0,1))).argmax(dim=1)
            if (preds_now == target).all():
                break

    with torch.no_grad():
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)

        preds_f_adv = f_net(adv_norm).argmax(dim=1)
        preds_R_adv = R_net(adv_norm).argmax(dim=1)
        preds_T_adv = T_net(adv_norm).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        mask_targeted = (preds_f_adv == target)
        n_success = mask_targeted.sum().item()
        if n_success > 0:
            f_target_success += n_success
            transfer_R += ((preds_R_adv == target) & mask_targeted).sum().item()
            transfer_T += ((preds_T_adv == target) & mask_targeted).sum().item()

# final metrics
clean_acc_f = 100.0 * correct_clean_f / total
clean_acc_R = 100.0 * correct_clean_R / total
clean_acc_T = 100.0 * correct_clean_T / total

adv_acc_f = 100.0 * correct_adv_f / total
adv_acc_R = 100.0 * correct_adv_R / total
adv_acc_T = 100.0 * correct_adv_T / total

transfer_rate_R = 100.0 * transfer_R / f_target_success if f_target_success>0 else float('nan')
transfer_rate_T = 100.0 * transfer_T / f_target_success if f_target_success>0 else float('nan')

print("\n=== Targeted C&W L2 (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")
print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")
print(f"\nAdv (Targeted C&W -> target={target_label}) accuracy (models evaluated on adv crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}%")
print(f"  R_net: {adv_acc_R:.2f}%")
print(f"  T_net: {adv_acc_T:.2f}%")
print(f"\nSuccess forcing label={target_label} on f_net: {f_target_success} / {total}  ({100*f_target_success/total:.2f}%)")
print(f"Of those forced-to-zero, transfer -> R_net: {transfer_rate_R:.2f}% | T_net: {transfer_rate_T:.2f}%")


In [ ]:

# ===============================================
# In This Cell:
# Visualization Cell: Display 10 clean test images with their predictions, 10 adversarial images generated by G with their predictions, and the corresponding differences

# ===============================================


import matplotlib.pyplot as plt
import torch

# Ensure models & G are in eval mode
f_net.eval()
R_net.eval()
G.eval()

# get one batch from test set
dataiter = iter(test_dataloader)
imgs_norm, labels = next(dataiter)
# ensure at least 10 available in the batch
n_show = min(10, imgs_norm.size(0))
imgs_norm = imgs_norm[:n_show].to(device)
labels = labels[:n_show].to(device)

# Convert normalized -> pixel [0,1] for generator
imgs_pixel = denormalize(imgs_norm)      # [0,1]

with torch.no_grad():
    # Generate perturbation in pixel space
    delta = G(imgs_pixel)                        # G outputs in [-1,1] (Tanh)
    delta = torch.clamp(delta, -eps, eps)        # bound in pixel space
    adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
    adv_norm = renormalize(adv_pixel)            # back to normalized space for models

    # Predictions on clean (normalized) and adv (normalized)
    preds_f_clean = f_net(imgs_norm).argmax(dim=1)
    preds_R_clean = R_net(imgs_norm).argmax(dim=1)
    preds_f_adv   = f_net(adv_norm).argmax(dim=1)
    preds_R_adv   = R_net(adv_norm).argmax(dim=1)

# Plot: 4 rows x n_show columns
plt.figure(figsize=(3*n_show, 10))

for i in range(n_show):
    # Row 1: Clean image (pixel space)
    ax = plt.subplot(4, n_show, 1 + i)
    img = imgs_pixel[i].cpu().permute(1,2,0).numpy()
    plt.imshow(img)
    ax.set_title(f"T:{labels[i].item()}\nR:{preds_R_clean[i].item()} f:{preds_f_clean[i].item()}", fontsize=9)
    ax.axis('off')

    # Row 2: Adversarial image (pixel space)
    ax = plt.subplot(4, n_show, n_show + 1 + i)
    adv_img = adv_pixel[i].cpu().permute(1,2,0).numpy()
    plt.imshow(adv_img)
    ax.set_title(f"R:{preds_R_adv[i].item()} f:{preds_f_adv[i].item()}", fontsize=9)
    ax.axis('off')

    # Row 3: Difference (grayscale) = adv - clean averaged over channels
    ax = plt.subplot(4, n_show, 2*n_show + 1 + i)
    diff = (adv_pixel[i] - imgs_pixel[i]).cpu().permute(1,2,0).mean(axis=2)
    plt.imshow(diff, cmap='gray')
    ax.set_title("Gray Δ", fontsize=9)
    ax.axis('off')

    # Row 4: Difference (seismic) with vmin/vmax
    ax = plt.subplot(4, n_show, 3*n_show + 1 + i)
    diff = (adv_pixel[i] - imgs_pixel[i]).cpu().permute(1,2,0).mean(axis=2)
    plt.imshow(diff, cmap='seismic', vmin=-eps, vmax=eps)
    ax.set_title("Seismic Δ", fontsize=9)
    ax.axis('off')

plt.suptitle("Row1: Clean (T/ R / f) | Row2: Adv (R / f) | Row3: Δ (gray) | Row4: Δ (seismic)", fontsize=14)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()
